# CogMem Phase 2 — Q-Value Guided LoRA Training\n\nTrain a LoRA adapter on llama3.2:3b using the 248 Q-valued memories from Phase 1.\nRuns locally on Paperspace A4000 (16GB VRAM) using QLoRA (4-bit).\n\nEstimated time: ~20-30 minutes.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# Cell 2: Install dependencies
!pip install torch transformers peft bitsandbytes datasets accelerate -q
!pip install pyyaml -q

In [ ]:
# Cell 3: Clone CogMem repo
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || (cd /notebooks/CogMem && git pull)
!ls /notebooks/CogMem/

In [ ]:
# Cell 4: Verify training data
# training.jsonl should be in the repo (push it before running)
import json
from pathlib import Path

JSONL_PATH = "/notebooks/CogMem/results/training.jsonl"

if not Path(JSONL_PATH).exists():
    print("ERROR: training.jsonl not found! Upload to /notebooks/CogMem/results/")
    raise FileNotFoundError(JSONL_PATH)

with open(JSONL_PATH) as f:
    lines = [json.loads(l) for l in f]

print(f"Training examples: {len(lines)}")
print(f"Sample:")
print(f"  User: {lines[0]['messages'][0]['content'][:100]}...")
print(f"  Assistant: {lines[0]['messages'][1]['content'][:150]}...")

from collections import Counter
unique = len(set(l['messages'][0]['content'] for l in lines))
print(f"Unique tasks: {unique}")
print(f"Avg Q-weighted copies: {len(lines)/unique:.1f}")

In [ ]:
# Cell 5: Train LoRA with QLoRA (4-bit) on A4000
import json
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
JSONL_PATH = "/notebooks/CogMem/results/training.jsonl"
ADAPTER_DIR = "/notebooks/CogMem/adapters/q_top_k"
LORA_RANK = 16
LORA_ALPHA = 32
EPOCHS = 3
LR = 1e-5

# Load data
raw_data = []
with open(JSONL_PATH) as f:
    for line in f:
        raw_data.append(json.loads(line))
print(f"Training samples: {len(raw_data)}")

# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

# LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenize
def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    tokenized = tokenizer(text, truncation=True, max_length=2048, padding=False)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
print(f"Tokenized dataset: {len(dataset)} examples")

# Train
training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=LR,
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
)

print("Starting LoRA training...")
trainer.train()

# Save
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

In [ ]:
# Cell 6: Package adapter for download
!tar czf /notebooks/cogmem_lora_adapter.tar.gz -C /notebooks/CogMem adapters/q_top_k/
!ls -lh /notebooks/cogmem_lora_adapter.tar.gz
print("Download cogmem_lora_adapter.tar.gz from Paperspace file browser")